## Análise — Perguntas do objetivo (Etapa 4.5)

**Fonte de dados:** tabelas gold (`workspace.gold.*`) do star schema CNO.

**Universo analítico** (definido em `docs/contexto-e-perguntas/definicoes.md`):
- Casas = áreas declaradas com `destinacao` em ('Residencial unifamiliar', 'Casa popular');
- somente área **Principal** (a casa em si) — `tipo_de_area = 'Principal'`;
- `categoria` em ('Existente', 'Obra Nova');
- métrica de tamanho: `metragem` (m²) — a silver já filtra `unidade_de_medida = m2`.

**Perguntas:**
- **P1.** Houve variação no tempo da área média das casas?
- **P2.** O tamanho populacional do município se relaciona com a área construída? E o da região geográfica imediata?
- **P3.** Distribuição das situações por porte de obra — obras menores têm mais chance de ficarem paralisadas/nulas?

**Alinhamento temporal P2:** população **no ano da obra** (join `fato_populacao.ano` = ano do início da obra). Obras em anos sem estimativa populacional (ex.: 1990, 2026+) ficam de fora do join.

In [0]:
%run ../ETL/notebooks/shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import adicionar_regiao

In [0]:
# Tabelas
FATO_OBRAS = 'workspace.gold.fato_obras'
FATO_POPULACAO = 'workspace.gold.fato_populacao'
DIM_DATA = 'workspace.gold.dim_data'
DIM_MUNICIPIO = 'workspace.gold.dim_municipio'
DIM_SITUACAO = 'workspace.gold.dim_situacao'
DIM_AREA = 'workspace.gold.dim_area'

# Universo analítico (definicoes.md)
DESTINACOES = ('Residencial unifamiliar', 'Casa popular')
CATEGORIAS = ('Existente', 'Obra Nova')

# P2: faixas de população (município / região imediata)
FAIXA_POP_MUNICIPIO = '''
CASE
  WHEN populacao < 20000 THEN 'Ate 20 mil'
  WHEN populacao < 50000 THEN '20 a 50 mil'
  WHEN populacao < 100000 THEN '50 a 100 mil'
  WHEN populacao < 200000 THEN '100 a 200 mil'
  WHEN populacao < 500000 THEN '200 a 500 mil'
  WHEN populacao < 1000000 THEN '500 mil a 1 milhao'
  ELSE 'Acima de 1 milhao'
END
'''
FAIXA_POP_REGIAO = '''
CASE
  WHEN populacao_regiao < 200000 THEN 'Ate 200 mil'
  WHEN populacao_regiao < 500000 THEN '200 a 500 mil'
  WHEN populacao_regiao < 1000000 THEN '500 mil a 1 milhao'
  WHEN populacao_regiao < 2000000 THEN '1 a 2 milhoes'
  WHEN populacao_regiao < 5000000 THEN '2 a 5 milhoes'
  ELSE 'Acima de 5 milhoes'
END
'''

# P3: faixas de metragem (70 m² = limiar de dispensa legal de registro no CNO)
FAIXA_METRAGEM = '''
CASE
  WHEN metragem < 50 THEN '0 - 50'
  WHEN metragem < 70 THEN '50 - 70'
  WHEN metragem < 100 THEN '70 - 100'
  WHEN metragem < 150 THEN '100 - 150'
  WHEN metragem < 250 THEN '150 - 250'
  WHEN metragem < 500 THEN '250 - 500'
  WHEN metragem < 1000 THEN '500 - 1.000'
  ELSE '> 1.000'
END
'''

In [0]:
# v_municipio: dim_municipio enriquecida com a coluna `regiao`
spark.table(DIM_MUNICIPIO).transform(adicionar_regiao).createOrReplaceTempView('v_municipio')

# v_casas: universo analítico (casas) pronto para as consultas das perguntas
df_casas = (
    spark.table(FATO_OBRAS)
    .join(
        spark.table(DIM_DATA).select('sk_data', 'ano'),
        on=F.col('sk_data_inicio') == F.col('sk_data'),
        how='inner',
    )
    .join(
        spark.table(DIM_AREA).select('sk_area', 'destinacao', 'tipo_de_area', 'categoria'),
        on='sk_area',
        how='inner',
    )
    .join(
        spark.table(DIM_SITUACAO).select('sk_situacao', 'descricao'),
        on='sk_situacao',
        how='inner',
    )
    .join(
        spark.table('v_municipio').select(
            'sk_municipio',
            'codigo_municipio',
            'sigla_uf',
            'regiao',
            'codigo_regiao_geografica_imediata',
        ),
        on='sk_municipio',
        how='inner',
    )
    .filter(F.col('destinacao').isin(*DESTINACOES))
    .filter(F.col('tipo_de_area') == 'Principal')
    .filter(F.col('categoria').isin(*CATEGORIAS))
    .select(
        'cno',
        'metragem',
        'ano',
        'sigla_uf',
        'regiao',
        'codigo_municipio',
        'codigo_regiao_geografica_imediata',
        F.col('descricao').alias('situacao_descricao'),
    )
)
df_casas.createOrReplaceTempView('v_casas')

print(f'Universo analítico (casas): {df_casas.count():,} linhas')

## P1 — Variação temporal da metragem média

**Como responder:** média (e mediana) da `metragem` das casas por ano/década de início da obra, em três níveis:
1. **Brasil** (todos os registros);
2. **Por região** (Norte, Nordeste, Centro-Oeste, Sudeste, Sul — derivada da UF);
3. **Por estado** (principais UFs — ajustável na constante `UF_FILTRO`).

**Discussão esperada:** há tendência de aumento/redução? A amostra por ano é suficiente nos anos iniciais? Considerar suavizar por década se houver poucos registros.

In [0]:
# P1.1a - Brasil: metragem media por ano
print('P1.1a - Brasil por ano')
display(spark.sql('''
SELECT
  ano,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY metragem), 2) AS metragem_mediana
FROM v_casas
GROUP BY ano
ORDER BY ano
'''))

Databricks visualization. Run in Databricks to view.

## Conclusão — P1

**Houve variação temporal na metragem média das casas registradas, mas os dados não permitem concluir que tenha ocorrido uma tendência contínua de aumento ou redução do tamanho das residências no Brasil.** Até 2018, observa-se uma trajetória de crescimento moderado da metragem média, acompanhada por uma mediana relativamente estável. A partir de 2019, entretanto, ocorre uma mudança estrutural na série, simultaneamente ao forte aumento do número de registros no CNO. Como o CNO substituiu o CEI e incorporou também registros anteriormente existentes no cadastro anterior, a composição e a cobertura da base mudam nesse período.

**Há ainda uma limitação importante relacionada à cobertura das obras de menor porte.** Determinadas construções residenciais unifamiliares de até **70 m²** — quando realizadas por pessoa física que não possui outro imóvel, destinadas a habitação popular ou econômica e executadas **sem utilização de mão de obra remunerada** — são dispensadas da obrigação de registro. Dessa forma, parte das residências de menor metragem pode não estar representada na base, **o que pode elevar a metragem média observada em relação à realidade**.

Além disso, a diferença entre média e mediana indica forte assimetria na distribuição das metragens, tornando a média sensível à presença de obras muito grandes.

**Portanto, não foi possível responder de forma conclusiva à P1.** As mudanças na cobertura e composição dos registros, somadas à possível sub-representação de residências menores, impedem afirmar, com base no CNO, que o tamanho das casas aumentou ou diminuiu ao longo do tempo.

## P2 — População × área construída

**Como responder:** comparar a `metragem` média das casas por **faixa de população**, usando a população **no ano da obra** (`fato_populacao.ano` = ano de início):
1. população do **município** da obra;
2. população da **região geográfica imediata**.

**Discussão esperada:** municípios/regiões mais populosos têm casas menores (adensamento, custo do terreno)? Analisar por faixas — cuidado com falácia ecológica.

In [0]:
# P2.1 - Faixas de populacao do municipio (ano da obra)
print('P2.1 - Metragem media por faixa de populacao do municipio')
display(spark.sql(f'''
SELECT
  {FAIXA_POP_MUNICIPIO} AS faixa_populacao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(c.metragem), 2) AS metragem_media
FROM v_casas c
JOIN {FATO_POPULACAO} p
  ON c.codigo_municipio = p.codigo_municipio AND c.ano = p.ano
GROUP BY faixa_populacao
ORDER BY MIN(p.populacao)
'''))

Databricks visualization. Run in Databricks to view.

In [0]:
# P2.2 - Faixas de populacao da regiao geografica imediata (ano da obra)
print('P2.2 - Metragem media por faixa de populacao da regiao imediata')
display(spark.sql(f'''
WITH pop_regiao AS (
  SELECT
    p.ano,
    m.codigo_regiao_geografica_imediata,
    SUM(p.populacao) AS populacao_regiao
  FROM {FATO_POPULACAO} p
  JOIN {DIM_MUNICIPIO} m ON p.sk_municipio = m.sk_municipio
  GROUP BY p.ano, m.codigo_regiao_geografica_imediata
)
SELECT
  {FAIXA_POP_REGIAO} AS faixa_populacao_regiao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(c.metragem), 2) AS metragem_media
FROM v_casas c
JOIN pop_regiao r
  ON c.codigo_regiao_geografica_imediata = r.codigo_regiao_geografica_imediata
 AND c.ano = r.ano
GROUP BY faixa_populacao_regiao
ORDER BY MIN(r.populacao_regiao)
'''))

Databricks visualization. Run in Databricks to view.

## Conclusão — P2

**Os dados indicam uma associação entre população e área construída.** Nas Regiões Geográficas Imediatas mais populosas, especialmente acima de 1 milhão de habitantes, observa-se aumento da metragem média das casas. Esse comportamento não evidencia a relação inversa esperada caso o adensamento e o custo do terreno fossem os principais fatores determinantes da metragem. Entretanto, a relação não é uniforme nas faixas menores e, por se tratar de uma análise agregada, não é possível estabelecer causalidade entre população e tamanho das residências.

## P3 — Situação da obra × porte

**Como responder:**
1. distribuição da `metragem` por situação da obra (`dim_situacao.descricao`);
2. taxa de obras `PARALISADA`/`NULA` por faixa de metragem (limiar de 70 m² = dispensa legal de registro).

**Discussão esperada:** obras menores estão mais sujeitas a paralisação/nulidade? A distribuição por situação é coerente com o ciclo de vida das obras?

In [0]:
# P3.1 - Distribuicao da metragem por situacao
print('P3.1 - Metragem por situacao da obra')
display(spark.sql('''
SELECT
  situacao_descricao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY metragem), 2) AS metragem_mediana
FROM v_casas
GROUP BY situacao_descricao
ORDER BY qtd_casas DESC
'''))

Databricks visualization. Run in Databricks to view.

In [0]:
# P3.2 - Taxa de PARALISADA/NULA por faixa de metragem
print('P3.2 - Obras paralisadas/nulas por faixa de metragem')
display(spark.sql(f'''
SELECT
  {FAIXA_METRAGEM} AS faixa_metragem,
  COUNT(*) AS qtd_casas,
  SUM(CASE WHEN situacao_descricao IN ('PARALISADA', 'NULA') THEN 1 ELSE 0 END) AS paralisada_nula,
  ROUND(
    100 * SUM(CASE WHEN situacao_descricao IN ('PARALISADA', 'NULA') THEN 1 ELSE 0 END) / COUNT(*),
    2
  ) AS pct_paralisada_nula
FROM v_casas
GROUP BY faixa_metragem
ORDER BY MIN(metragem)
'''))

Databricks visualization. Run in Databricks to view.

## Conclusão — P3

**Os dados não indicam que obras menores tenham maior chance de ficarem paralisadas ou nulas.** Na base analisada, a proporção de obras `PARALISADA` ou `NULA` aumenta conforme a metragem, especialmente acima de 150 m². Entretanto, a possível sub-representação de construções menores que podem ser dispensadas de registro limita a generalização desse resultado para o universo de todas as casas construídas. A diferença entre média e mediana também indica que obras de grande porte influenciam significativamente as estatísticas de algumas situações.

## Discussão geral

A análise apresenta um panorama das casas **residenciais unifamiliares e populares registradas no CNO**, mas evidencia limitações importantes de representatividade. Na **P1**, não foi possível concluir se o tamanho das casas aumentou ou diminuiu ao longo do tempo, devido às mudanças na cobertura do cadastro e à possível sub-representação de construções menores. Na **P2**, observou-se associação entre população e metragem média, especialmente nas regiões mais populosas, mas sem evidência de causalidade. Na **P3**, a hipótese de maior ocorrência de paralisação ou nulidade entre obras menores não foi confirmada.

Os resultados devem ser interpretados considerando que o **CNO possui finalidade cadastral e tributária**, podendo não contemplar obras informais ou determinadas construções de até **70 m²** dispensadas de registro. Na P2, o alinhamento da população ao **ano de início da obra** garante maior coerência temporal, mas exclui anos sem estimativa populacional disponível, como **1990 e 2026+**.

**Assim, os resultados caracterizam o universo de obras registradas no CNO, mas não podem ser generalizados diretamente para todas as casas construídas no Brasil.**